# RMY Generation on Google Colab

This notebook is the main end-to-end test of the `rmy` package. For every city in `data/cities/` it:

1. checks the input weather files,
2. detects heat waves and cold spells (thresholds, GNN and EVT),
3. builds the RMY weather file,
4. compares the RMY's extremes with the base TMY,
5. plots every detected event (Extreme Event Explorer),

and then zips all outputs for download.

Input folders (8 US cities are included: Boston, Chicago, Dulles, Las Vegas NM, Los Angeles, Phoenix, San Francisco, Seattle):

```
data/cities/
└── <City>/
    ├── AMY/   AMY EPWs, one per year (the year must be in the file name)
    └── TMY/   exactly one base TMY EPW
```

Start with **Settings** below: choose one city or **All**. To run your own city, type its name there (or use section 2.1): you will be asked to upload its AMYs and TMY, which go into a new `data/cities/<YourCity>/` folder.

Outputs, in `/content/output/<City>/`:

```
hotspells/                heat-wave events and yearly statistics (CSV)
coldspells/               cold-spell events and yearly statistics (CSV)
<TMY file name>.epw       the RMY weather file, with TMY/TMYx in the name replaced by RMY
event_timeline.png        the Extreme Event Explorer plot
log.txt                   the full run log
```

# **Settings**

Choose what to run, then use **Runtime → Run all**.

In [ ]:
# @title Settings { display-mode: "form" }
# @markdown **City**: one of the folders in `data/cities`, or **All** to run every city. You can also type the name of your own city; section 2.1 then asks you to upload its files.
CITY = "All"  # @param ["All", "Boston", "Chicago", "Dulles_Washington", "Las_Vegas", "Los_Angeles", "Phoenix", "San_Francisco", "Seattle"] {allow-input: true}
# @markdown Run the package tests after cloning (adds about 2 minutes).
RUN_TESTS = False  # @param {type:"boolean"}

# **0.0 Dependencies**

In [ ]:
import importlib.util, subprocess, sys

RUN_TESTS = globals().get("RUN_TESTS", False)   # default if Settings was not run
needed = ["pandas", "numpy", "scipy", "matplotlib"] + (["pytest"] if RUN_TESTS else [])
missing = [p for p in needed if importlib.util.find_spec(p) is None]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)
print("Installed:", ", ".join(missing) if missing else "nothing (all packages already available)")

# **1.0 Clone the repository**

A fast clone: only the latest version, and only the folders this notebook needs (the published RMY/FRMY files, README images and git history are skipped). `BRANCH` selects the version of the code: `main` is the released version.

In [ ]:
BRANCH = "main"
RUN_TESTS = globals().get("RUN_TESTS", False)
SPARSE_DIRS = "rmy examples data/cities" + (" tests data/RMY" if RUN_TESTS else "")

%cd /content
!rm -rf RMY
!git clone -q --depth 1 --single-branch --branch {BRANCH} --filter=blob:none --sparse https://github.com/Nadatarkhan/RMY.git
%cd /content/RMY
!git sparse-checkout set {SPARSE_DIRS}
!git log -1 --format='Running commit %h: %s'

Package tests, if `RUN_TESTS` is ticked in Settings. They should all pass before you trust the output.

In [ ]:
if globals().get("RUN_TESTS", False):
    !python -m pytest -q tests/
else:
    print("Tests skipped (tick RUN_TESTS in Settings to run them).")

# **2.0 Cities to run**

Uses `CITY` from Settings: **All** runs every folder in `data/cities`, a city name runs that city.

In [ ]:
from pathlib import Path

CITIES_DIR = Path("/content/RMY/data/cities")
OUTPUT_DIR = Path("/content/output")

CITY = globals().get("CITY", "All")   # default if Settings was not run
AVAILABLE = sorted(p.name for p in CITIES_DIR.iterdir() if (p / "AMY").is_dir() and (p / "TMY").is_dir())
if CITY == "All":
    CITIES = AVAILABLE
elif CITY in AVAILABLE:
    CITIES = [CITY]
else:
    CITIES = []
    print(f"There is no data/cities/{CITY} folder yet: section 2.1 will ask you to upload its files.")
print("Available cities:", ", ".join(AVAILABLE))
print("Cities to run:", ", ".join(CITIES) or "none yet")

## 2.1 Optional: add your own city

If you typed a new city name in Settings, this cell uses it. You can also set `MY_CITY` here instead. When asked, first select all of your AMY EPW files (one per year, with the year in each file name), then the one base TMY EPW. They are saved to `data/cities/<MY_CITY>/AMY/` and `data/cities/<MY_CITY>/TMY/`.

With `ONLY_MY_CITY = True` the rest of the notebook runs just your city; set it to `False` to run it together with the cities chosen in Settings.

In [ ]:
MY_CITY = None        # e.g. "Toronto"; None uses a new city typed in Settings
ONLY_MY_CITY = True

if MY_CITY is None and CITY != "All" and CITY not in AVAILABLE:
    MY_CITY = CITY

if MY_CITY:
    from google.colab import files
    city_dir = CITIES_DIR / MY_CITY
    for sub, prompt in (("AMY", "Select ALL of your AMY EPW files (one per year)"),
                        ("TMY", "Select your ONE base TMY EPW file")):
        folder = city_dir / sub
        folder.mkdir(parents=True, exist_ok=True)
        for old in folder.glob("*.epw"):
            old.unlink()
        print(prompt)
        for name, content in files.upload().items():
            (folder / name).write_bytes(content)
        print(f"  {sub}: {len(list(folder.glob('*.epw')))} file(s) in {folder}")
    if len(list((city_dir / "TMY").glob("*.epw"))) != 1:
        raise ValueError("The TMY folder must hold exactly one EPW file.")
    CITIES = [MY_CITY] if ONLY_MY_CITY else sorted(set(CITIES) | {MY_CITY})
    print("Cities to run:", ", ".join(CITIES))

# **3.0 Check the input files**

Lists invalid values (netCDF fill values such as `9.96921e+36`, out-of-range numbers such as a 447 m/s wind speed) in every input file. The pipeline treats them as missing data, but files with many of them should be re-sourced before you trust the RMY.

In [ ]:
if not CITIES:
    raise ValueError("No city selected: choose one in Settings, or add your own in section 2.1.")

from rmy.validation import audit_epws, print_report

for city in CITIES:
    files = sorted((CITIES_DIR / city / "TMY").glob("*.epw")) + sorted((CITIES_DIR / city / "AMY").glob("*.epw"))
    print_report(audit_epws(files), f"{city} ({len(files)} files)", show_missing=False)

# **4.0 Generate the RMYs**

The full log for each city is saved to `OUTPUT_DIR/<City>/log.txt`; the key lines are printed here. The table at the end summarises every city.

In [ ]:
import contextlib, io
import pandas as pd
from rmy import run_full_rmy_pipeline
from rmy.utils import list_epws, read_epw_file

KEY_LINES = ("thresholds (all", "GNN:", "EVT ", "runner-up", "inserted as its most", "events integrated",
             "Month ", "smoothing:", "WARNING", "Final RMY")
rows, RMY_PATHS = [], {}
for city in CITIES:
    print(f"\n=== {city} ===")
    log = io.StringIO()
    with contextlib.redirect_stdout(log):
        rmy_path = run_full_rmy_pipeline(CITIES_DIR / city / "AMY", CITIES_DIR / city / "TMY", OUTPUT_DIR / city)
    (OUTPUT_DIR / city / "log.txt").write_text(log.getvalue())
    RMY_PATHS[city] = rmy_path
    for line in log.getvalue().splitlines():
        if any(key in line for key in KEY_LINES):
            print(line.strip())

    out = OUTPUT_DIR / city
    heat = pd.read_csv(out / "hotspells/heatwave_events_peak.csv")
    cold_path = out / "coldspells/coldspells_events_peak.csv"
    cold = pd.read_csv(cold_path) if cold_path.exists() else pd.DataFrame(columns=["duration"])
    base = read_epw_file(list_epws(CITIES_DIR / city / "TMY")[0])
    rmy = read_epw_file(rmy_path)
    dev = rmy.groupby("month").temp_air.mean() - base.groupby("month").temp_air.mean()
    rows.append({
        "city": city,
        "heat peak year": int(pd.read_csv(out / "hotspells/heatwave_stats_peak.csv").year[0]),
        "heat events in peak year": len(heat),
        "longest heat event (days)": int(heat.duration.max()) if len(heat) else 0,
        "cold peak year": int(pd.read_csv(out / "coldspells/coldspells_stats_peak.csv").year[0]),
        "cold events in peak year": len(cold),
        "longest cold event (days)": int(cold.duration.max()) if len(cold) else 0,
        "annual mean vs TMY (°C)": round(rmy.temp_air.mean() - base.temp_air.mean(), 2),
        "largest monthly change (°C)": f"{dev[dev.abs().idxmax()]:+.1f} (month {dev.abs().idxmax()})",
        "RMY file": rmy_path.name,
    })

summary = pd.DataFrame(rows)
summary.to_csv(OUTPUT_DIR / "summary.csv", index=False)
display(summary)

# **4.5 Temperature extremes: RMY vs. TMY**

Coldest and hottest hour of each new RMY against its base TMY (Δ = RMY − TMY).

In [ ]:
import pandas as pd
from rmy.utils import list_epws, read_epw_file

rows = []
for city in CITIES:
    tmy = read_epw_file(list_epws(CITIES_DIR / city / "TMY")[0]).temp_air
    rmy = read_epw_file(RMY_PATHS[city]).temp_air
    rows.append({"City": city, "TMY min": tmy.min(), "RMY min": rmy.min(), "Δ min": rmy.min() - tmy.min(),
                 "TMY max": tmy.max(), "RMY max": rmy.max(), "Δ max": rmy.max() - tmy.max()})

display(pd.DataFrame(rows).round(1))

# **5.0 Visualization: Extreme Event Explorer**

One plot per city. Each circle is one detected event at its start date; its size is the duration. Heat waves are colored by average heat index (°C), cold spells by average wind speed.

`SHOW = "all"` plots every AMY year; `SHOW = "peak"` plots only the events inserted into the RMY.

In [ ]:
from rmy.visualization import visualize_extreme_events

SHOW = "all"   # or "peak"
suffix = "" if SHOW == "all" else "_peak"

for city in CITIES:
    out = OUTPUT_DIR / city
    visualize_extreme_events(
        out / f"hotspells/heatwave_events{suffix}.csv",
        out / f"coldspells/coldspells_events{suffix}.csv",
        output_path=out / f"event_timeline{suffix}.png",
        title=f"{city}: heat waves and cold spells",
    )

# **6.0 Download the results**

In [ ]:
import shutil
from google.colab import files

archive = shutil.make_archive("/content/rmy_output", "zip", OUTPUT_DIR)
files.download(archive)